# Tutorial: End-to-End Orchestration Trace and Checkpoint Analysis

Audience:
- Agent architects validating runtime protocol guarantees.
- AI engineers debugging execution flow and failure recovery.
- Data scientists analyzing plan/execution coupling in real runs.

Prerequisites:
- Notebook 1 and 2 completed.
- Optional API key for live end-to-end run.

Learning goals:
- Execute ROMA end-to-end and inspect the resulting DAG.
- Compare recursive and event-driven orchestration modes.
- Inspect checkpoint artifacts and extract execution evidence.


## Outline

1. Setup and config loading.
2. Define reproducible execution harness.
3. Run recursive orchestration (optional live call).
4. Inspect DAG/tasks/dependencies from solver memory.
5. Inspect checkpoint files as persisted execution trace.
6. Compare recursive vs event-driven mode.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "roma_dspy").exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / "src" / "roma_dspy").exists():
            REPO_ROOT = candidate
            break

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH added:", SRC_PATH)


REPO_ROOT: /Users/admin/TuanDung/repos/ROMA
SRC_PATH added: /Users/admin/TuanDung/repos/ROMA/src


In [2]:
import asyncio
import gzip
import json
import os
from pathlib import Path

import networkx as nx

from roma_dspy.config.manager import ConfigManager
from roma_dspy.core.engine.solve import RecursiveSolver
from roma_dspy.types import TaskStatus

cfg_manager = ConfigManager(config_dir=REPO_ROOT / "config")
cfg_path = REPO_ROOT / "config" / "local" / "openai_config.yaml"
if not cfg_path.exists():
    cfg_path = REPO_ROOT / "config" / "defaults" / "config.yaml"

config = cfg_manager.load_config(config_path=str(cfg_path))
print("Config path:", cfg_path)
print("Runtime max_depth:", config.runtime.max_depth)
print("Runtime max_concurrency:", config.runtime.max_concurrency)


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/opt/anaconda3/envs/vllm/lib/python3.12/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
            ~~~~~~~~~~~~~~~~~^^^^^^^^^^^
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/vllm/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 575, in _log_error
    f.result()
  File "/opt/anaconda3/envs/vllm/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 347, in dispatch_control
    async with self._control_lock:
               ^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/vllm/lib/python3.12/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should encode the Optional
                     ^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/vllm/lib/python

Config path: /Users/admin/TuanDung/repos/ROMA/config/local/openai_config.yaml
Runtime max_depth: 5
Runtime max_concurrency: 5


## Execution Harness

We keep one helper to run either orchestration mode and return:
- root task result
- in-memory `TaskDAG`
- task-level summary snapshot


In [3]:
async def run_execution(task: str, *, mode: str = "recursive", max_depth: int = 2, concurrency: int = 2):
    solver = RecursiveSolver(config=config, max_depth=max_depth)

    if mode == "recursive":
        result = await solver.async_solve(task)
    elif mode == "event":
        result = await solver.async_event_solve(task, concurrency=concurrency)
    else:
        raise ValueError("mode must be 'recursive' or 'event'")

    dag = solver.last_dag
    all_tasks = dag.get_all_tasks(include_subgraphs=True) if dag else []
    summary = {
        "execution_id": dag.execution_id if dag else None,
        "mode": mode,
        "status": str(result.status),
        "result_preview": (result.result or "")[:300],
        "task_count": len(all_tasks),
        "completed": sum(1 for t in all_tasks if t.status == TaskStatus.COMPLETED),
        "failed": sum(1 for t in all_tasks if t.status == TaskStatus.FAILED),
    }
    return solver, result, dag, summary


## Live End-to-End Run (Optional)

Set `RUN_LIVE_EXECUTION = True` for real model inference.
If skipped, you can still inspect existing checkpoints from prior runs.


In [4]:
RUN_LIVE_EXECUTION = False  # Set True to run an actual execution
TASK = "Analyze whether remote work improves engineering productivity. Return 3 pros, 3 cons, and one balanced recommendation."

if RUN_LIVE_EXECUTION:
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not set; skipping live execution.")
    else:
        solver, result, dag, run_summary = asyncio.run(
            run_execution(TASK, mode="recursive", max_depth=2)
        )
        print("Run summary:", run_summary)
else:
    print("Live execution skipped. Toggle RUN_LIVE_EXECUTION=True to execute.")


Live execution skipped. Toggle RUN_LIVE_EXECUTION=True to execute.


## Inspect DAG And Dependency Structure

This cell works if a live run was just executed in-memory.


In [5]:
if "dag" not in globals() or dag is None:
    print("No in-memory DAG available. Run live execution first.")
else:
    print("DAG id:", dag.dag_id)
    print("Execution id:", dag.execution_id)
    print("DAG stats:", dag.get_statistics())

    print()
    print("Tasks:")
    for t in dag.get_all_tasks(include_subgraphs=True):
        print({
            "task_id": t.task_id[:8],
            "depth": t.depth,
            "status": str(t.status),
            "task_type": str(t.task_type),
            "node_type": str(t.node_type) if t.node_type else None,
            "deps": [d[:8] for d in t.dependencies],
            "goal": t.goal[:90],
        })


No in-memory DAG available. Run live execution first.


## Read Persisted Checkpoints (Real Trace Artifact)

Checkpoint files provide durable evidence for:
- planner outputs,
- task graph structure,
- terminal results,
- recovery metadata.


In [6]:
from pathlib import Path
import gzip
import json


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "roma_dspy").exists() or (candidate / ".git").exists():
            return candidate
    return start

repo_root = globals().get("REPO_ROOT")
if repo_root is None:
    repo_root = _find_repo_root(Path.cwd())
else:
    repo_root = Path(repo_root)

checkpoint_dir = repo_root / ".checkpoints"
if not checkpoint_dir.exists():
    print("Checkpoint directory not found:", checkpoint_dir)
else:
    checkpoint_files = sorted(
        checkpoint_dir.glob("checkpoint_*.json.gz"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    print("Checkpoint count:", len(checkpoint_files))

    if not checkpoint_files:
        print("No checkpoints found.")
    else:
        latest = checkpoint_files[0]
        print("Latest checkpoint:", latest.name)

        try:
            with gzip.open(latest, "rt", encoding="utf-8") as f:
                ckpt = json.load(f)
        except Exception as exc:
            print("Failed to read checkpoint:", type(exc).__name__, str(exc))
            ckpt = None

        if ckpt:
            ckpt_root_dag = ckpt.get("root_dag", {})
            tasks = ckpt_root_dag.get("tasks", {})

            print({
                "checkpoint_id": ckpt.get("checkpoint_id"),
                "execution_id": ckpt.get("execution_id"),
                "trigger": ckpt.get("trigger"),
                "state": ckpt.get("state"),
                "tasks_total": len(tasks),
                "completed": len(ckpt_root_dag.get("completed_tasks", [])),
                "failed": len(ckpt_root_dag.get("failed_tasks", [])),
            })

            root_candidates = [t for t in tasks.values() if t.get("depth") == 0]
            if root_candidates:
                root_task = root_candidates[0]
                print()
                print("Root task preview:")
                print("goal:", root_task.get("goal", "")[:180])
                print("status:", root_task.get("status"))
                print("result:", (root_task.get("result") or "")[:400])


Checkpoint count: 29
Latest checkpoint: checkpoint_20260224_132533_582418_e2eee04d.json.gz
{'checkpoint_id': 'checkpoint_20260224_132533_582418_e2eee04d', 'execution_id': '2fd8406b-1e56-4c50-9123-0777a7e34971', 'trigger': 'execution_complete', 'state': 'valid', 'tasks_total': 14, 'completed': 14, 'failed': 0}

Root task preview:
goal: Analyze whether remote work improves engineering productivity. Return: 3 pros, 3 cons, and a balanced recommendation.
status: COMPLETED
result: **Pros of Remote Work on Engineering Productivity:**
1. **Increased Flexibility:** Engineers can work during their most productive hours.
2. **Reduced Commute Time:** Saving time and energy that can be redirected to work.
3. **Improved Work-Life Balance:** Enhances job satisfaction and motivation.

**Cons of Remote Work on Engineering Productivity:**
1. **Communication Challenges:** Remote work ca


## Compare Recursive vs Event-Driven Orchestration

Run both modes on the same task and compare shape/latency externally.
This cell does not auto-run by default to avoid extra model cost.


In [7]:
RUN_MODE_COMPARISON = False
COMPARE_TASK = "Draft a concise engineering decision memo: when to use remote, hybrid, or onsite mode for backend teams."

if RUN_MODE_COMPARISON:
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not set; cannot run comparison.")
    else:
        _, _, _, recursive_summary = asyncio.run(run_execution(COMPARE_TASK, mode="recursive", max_depth=2, concurrency=2))
        _, _, _, event_summary = asyncio.run(run_execution(COMPARE_TASK, mode="event", max_depth=2, concurrency=2))
        print("Recursive summary:", recursive_summary)
        print("Event-driven summary:", event_summary)
else:
    print("Comparison skipped. Toggle RUN_MODE_COMPARISON=True to execute both modes.")


Comparison skipped. Toggle RUN_MODE_COMPARISON=True to execute both modes.


## Exercises

1. Force smaller `max_depth` and observe when `force_execute_async` is triggered.
2. Increase `concurrency` in event mode and inspect whether dependency bottlenecks still dominate.
3. Build a custom plan-quality dashboard by joining planner metrics (Notebook 2) with final execution outcomes.
